In [ ]:
!pip install indictranstoolkit sacrebleu sentencepiece transformers safetensors

In [ ]:
# ── Model Loading ─────────────────────────────────────────────────────────────
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor
from safetensors.torch import load_file

# Install dependencies
# pip install indictranstoolkit sacrebleu sentencepiece transformers safetensors

MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"
WEIGHTS_PATH = "path/to/model.safetensors"  # replace with your HuggingFace model path

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Load base architecture
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

# Load fine-tuned weights
state_dict = load_file(WEIGHTS_PATH)
model.load_state_dict(state_dict, strict=False)
model = model.to("cuda").eval()

ip = IndicProcessor(inference=True)
print("Model loaded succesfsfully")



In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────
def translate(sentences, src_lang="eng_Latn", tgt_lang="kas_Arab", batch_size=32):
    all_translations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        preprocessed = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)
        inputs = tokenizer(
            preprocessed, return_tensors="pt",
            padding=True, truncation=True, max_length=256
        ).to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs, num_beams=5, max_length=256,
                length_penalty=1.0, early_stopping=True
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_translations.extend(ip.postprocess_batch(decoded, lang=tgt_lang))
    return all_translations

# Example usage
sentences = ["She was a true visionary.", "I go to my school daily."]
translations = translate(sentences)
for src, tgt in zip(sentences, translations):
    print(f"{src} → {tgt}")

In [ ]:
#Generate submission files
import pandas as pd
from tqdm import tqdm

test_df = pd.read_csv("englishdev.csv")

all_translations = []
sentences = test_df["sentence"].tolist()
for i in tqdm(range(0, len(sentences), 32)):
    batch = sentences[i : i + 32]
    all_translations.extend(translate(batch))

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "kashmiri_text": all_translations
})
submission.to_csv("submission.csv", index=False)
print(f"Saved {len(submission)} rows to submission.csv")
print(submission.head(5))